# M1-INF4 – Python Workshop for Data Analysis
## Chapter 2 – Data Manipulation with pandas

**M1 IBI / SBI — Paris 1 Panthéon-Sorbonne**

### Learning objectives

By the end of this chapter, students will be able to:

- Understand pandas `DataFrame` and `Series` objects
- Inspect and explore a dataset
- Select, filter, and sort data
- Create and transform columns
- Compute summary statistics
- Aggregate data using `groupby()`
- Manipulate indexes and subsets
- Import and export tabular data

## 1. Introducing DataFrames

`pandas` is a Python library designed for data manipulation and analysis.
Its central data structure is the **DataFrame**, a two-dimensional table
organized into rows and columns.

Some of the first methods we use when discovering a dataset are:

- `head()` — display the first rows
- `info()` — inspect columns, data types, and missing values
- `describe()` — compute descriptive statistics

In [ ]:
import pandas as pd

data = {
    "Name": ["Fido", "Buddy", "Rex"],
    "Age": [4, 5, 6]
}

df = pd.DataFrame(data)

df.head()

### From NumPy to pandas
A NumPy array gives you numerical operations; pandas adds **column names, row labels and heterogeneous column types**.
A `Series` is one labelled column. A `DataFrame` is a table containing several columns.
Run the examples, then complete the exercises in the empty cells. Explain what each result means, not only which method you used.

In [ ]:
print(type(df["Age"]))
print(type(df[["Age"]]))
df[["Name", "Age"]]

## 2. Inspecting real data
We start with `homelessness.csv`, supplied with DataCamp's *Data Manipulation with pandas* course.
Each row represents one US state or the District of Columbia. The file contains counts of individuals experiencing homelessness, people in families experiencing homelessness, state population and region.
These are observations in a supplied dataset, **not current estimates**. An absolute count and a rate answer different questions.

Keep the `notebooks/` and `data/` folders together. Open this notebook from `notebooks/`.

In [ ]:
homelessness = pd.read_csv("../data/homelessness.csv")

homelessness.head()

The CSV also contains an exported index column (`Unnamed: 0`). We remove this technical column explicitly; it is not a measured variable.

In [ ]:
homelessness = homelessness.drop(columns=["Unnamed: 0"], errors="ignore")
print("Shape:", homelessness.shape)
print("Columns:", homelessness.columns.tolist())
homelessness.info()

In [ ]:
homelessness.describe()

### Your turn — Understand the table
1. Display the last three rows.
2. Count the missing values in each column.
3. How many distinct regions are represented?
4. In a Markdown cell, explain what one row represents and why `state_pop` should not be added to the homelessness counts.

In [ ]:
# Your code here

## 3. Selecting, filtering and sorting
Select a column with `table["column"]`; select several columns with a list of names.
A comparison creates a Boolean mask, just as with NumPy. `.loc[rows, columns]` lets you choose both rows and columns.

In [ ]:
homelessness[["state", "individuals"]].head()

In [ ]:
mask = homelessness["region"] == "Mountain"
homelessness.loc[mask, ["state", "individuals", "state_pop"]]

Combine conditions with `&` (and), `|` (or), and `~` (not). Put each comparison in parentheses; do not use Python's `and` between Series.

In [ ]:
homelessness.loc[
    (homelessness["region"] == "Mountain") & (homelessness["individuals"] > 1000),
    ["state", "individuals"]
]

In [ ]:
homelessness.sort_values("individuals", ascending=False)[["state", "individuals"]].head(5)

`head(5)` keeps the first five rows of the current order. `.iloc` selects by position; `.loc` selects by labels or Boolean masks. Neither sorting nor selecting changes the original table unless you assign the result.

In [ ]:
homelessness.iloc[:3, :2]

### Your turn — A targeted selection
Select states with more than 5 million inhabitants and more than 5,000 individuals experiencing homelessness.
Display only `state`, `state_pop` and `individuals`, ordered by `individuals` from largest to smallest.
Save your result in a variable called `selected_states`.

In [ ]:
# Your code here

## 4. Creating meaningful indicators
Vectorized calculations work on entire columns. There is no need to loop over rows.
Here is a rate for **individuals only**; people in families are not included in this indicator.

In [ ]:
homelessness["individuals_per_10k"] = homelessness["individuals"] / homelessness["state_pop"] * 10_000
homelessness[["state", "individuals_per_10k"]].head().round(2)

### Group problem 1 — Where should an analyst look first?
A public-policy team asks: **“Which states have the greatest need?”** Your group must show why the answer depends on the indicator.

1. On a copy named `needs`, create `total_homeless` by adding `individuals` and `family_members`.
2. Create `total_per_10k`, using the state population as denominator.
3. Build two top-five tables: one by total count and one by rate. Include state names and the indicator values.
4. Identify the states common to both lists (manual comparison is acceptable).
5. Write a short recommendation: which ranking answers a question about the number of people to support? Which answers a question about prevalence? Explain one limitation of using these figures alone to allocate funding.

**Work as a group.** Agree on your indicators before coding. Each member must be able to explain the denominator and reproduce the filtering/sorting steps. Keep your code and a three-sentence interpretation below. No solution is included in this notebook.

In [ ]:
# Group problem 1 — your code here

**Group interpretation:**

*Write your recommendation here.*

## 5. Summarising data with pandas
A summary reduces many observations to a few numbers. Choose the variable and the question first.
`count()` counts non-missing values; `size()` counts rows. `value_counts()` counts occurrences of each category.

In [ ]:
homelessness["individuals"].agg(["sum", "mean", "median", "max"])

In [ ]:
homelessness["region"].value_counts()

### From a single summary to one summary per group
`groupby()` follows three steps: split rows into groups, apply a calculation, and combine the results.
The following example sums counts within each region. It does not average state rates.

In [ ]:
regional = homelessness.groupby("region")[["individuals", "family_members", "state_pop"]].sum()
regional

The group labels now form the index. `reset_index()` turns those labels back into a regular column.

In [ ]:
regional = regional.reset_index()
regional.head()

### Your turn — Compare regions fairly
Using `regional`, create the total homelessness count and the total rate per 10,000 inhabitants for each region.
Sort by rate, highest first. Explain why you divide the **sum of counts by the sum of populations**, rather than taking the simple average of state rates.

In [ ]:
# Your code here

## 6. Transfer your skills to a business dataset
The second file, `sales_subset.csv`, is a sample supplied with the same DataCamp course.
One row describes a store–department–date observation; `weekly_sales` records weekly sales in US dollars. The file is a subset: do not assume complete time coverage or equal numbers of observations per store.

Main columns: `store` (store identifier), `type` (store category), `department`, `date`, `weekly_sales`, `is_holiday`. Other columns describe temperature, fuel price and unemployment.
Negative sales occur in the file. Their cause is not established here: keep them for this exercise and report their presence rather than silently removing them.

In [ ]:
sales = pd.read_csv("../data/sales_subset.csv", index_col=0)
sales.head()

### Group problem 2 — Does the biggest total mean the best performance?
A retail manager claims: **“The store type with the highest total sales performs best.”** Investigate this claim using the supplied sample.

1. Inspect the table: report its dimensions, number of distinct stores and number of observations with negative `weekly_sales`.
2. For each store type, calculate total sales, mean sales per observation and number of observations. You may use separate `groupby()` operations or `.agg()`.
3. Determine whether the same store type leads for total and mean sales.
4. For each store, calculate total sales and number of observations. Display the five stores with the largest totals.
5. Write a four-sentence response to the manager. Explain how unequal sample sizes affect totals and why a mean per observation is not a mean per store or proof of profitability.
6. Save the store-type summary as `sales_by_type.csv` in the notebook folder, with `index=False` (after `reset_index()` if necessary).

**Group deliverable:** reproducible code, the summary tables and your written response. Split the work, then review the whole answer together. Each member must be able to explain all calculations. No solution is included here.

In [ ]:
# Group problem 2 — inspection and calculations

**Group interpretation:**

*Write your response to the manager here.*

## 7. Check your understanding
Before moving on, make sure you can:
- explain the difference between a Series and a DataFrame;
- select columns and combine filtering conditions;
- sort and build a vectorized indicator;
- distinguish a count, a sum, a mean and a rate;
- use `groupby()` and export a table;
- explain the unit represented by a row before interpreting results.

Restart the kernel and run the notebook from top to bottom. Your results should not depend on running cells in a special order.

**Next:** investigate missing values, inconsistent types and duplicates more systematically.

### Sources and teaching adaptation
Datasets: DataCamp, *Data Manipulation with pandas*, supplied course-resource files `homelessness.csv` and `sales_subset.csv`.
This notebook extends the existing M1-INF4 pandas introduction. Its question-led group activities draw on the teaching approach of Anthony's previous course materials. The activities and explanations added here are an original adaptation, not a reproduction of the paid DataCamp exercises or the private cinema dataset.